# EDA 003.05 — Principal Component Analysis (PCA)

**Kaggle Feature Engineering Course — Lesson 5**
**Reference:** [Principal Component Analysis](https://www.kaggle.com/code/ryanholbrook/principal-component-analysis) by Ryan Holbrook

---

## Learning Objectives

By the end of this notebook you will understand:

1. What PCA does mathematically — finding **axes of maximum variance**
2. How to use PCA for **dimensionality reduction**
3. How to use PCA components as **new features** (not just for compression)
4. How to interpret the **loadings** of each component
5. How to choose the **number of components** using explained variance
6. When PCA helps vs when it doesn't

---

## What Is PCA?

> **Principal Component Analysis (PCA)** finds new axes (principal components) that are linear combinations of the original features, ordered by the amount of variance they explain.

```
Original data (correlated features):
    X₁, X₂, X₃, ...  →  many dimensions, some redundant

After PCA:
    PC1  (explains most variance)
    PC2  (explains next most, uncorrelated with PC1)
    PC3  ...
```

### Two uses in feature engineering:

1. **Dimensionality reduction** — replace N correlated features with k < N components, retaining most variance
2. **Feature creation** — use PCA components (especially PC1) as new informative features, then engineer from them

> **Important:** Always **standardise features** before PCA (mean=0, std=1). PCA is scale-sensitive — features with large variance will dominate.

---

## The Mathematics (Intuition)

PCA finds vectors (eigenvectors of the covariance matrix) that point in the directions of maximum variance:

$$\text{Covariance matrix} \quad \Sigma = \frac{1}{n} X^T X$$

$$\Sigma \mathbf{v} = \lambda \mathbf{v}$$

- **Eigenvectors** $\mathbf{v}$ → directions of the new axes (principal components)
- **Eigenvalues** $\lambda$ → amount of variance explained by each component

The **explained variance ratio** for PC$k$ is $\lambda_k / \sum_i \lambda_i$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Demo 1 — PCA in 2D (Visualising the New Axes)

We start with two correlated features and show how PCA finds new orthogonal axes aligned with the data's variance.

In [ ]:
np.random.seed(42)
# Two correlated features
x1 = np.random.normal(0, 2, 200)
x2 = 0.8 * x1 + np.random.normal(0, 0.8, 200)
X_2d = np.column_stack([x1, x2])

pca_2d = PCA(n_components=2)
X_pca  = pca_2d.fit_transform(X_2d)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Original space with PCA axes overlaid
axes[0].scatter(x1, x2, alpha=0.4, s=20)
origin = np.mean(X_2d, axis=0)
for i, color in enumerate(['red', 'blue']):
    vec = pca_2d.components_[i] * pca_2d.explained_variance_[i]**0.5 * 2
    axes[0].annotate('', xy=origin + vec, xytext=origin,
                     arrowprops=dict(arrowstyle='->', color=color, lw=2))
axes[0].set_title(f'Original space (corr = {np.corrcoef(x1,x2)[0,1]:.2f})', fontsize=12)
axes[0].set_xlabel('x1'); axes[0].set_ylabel('x2')

# PCA space
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.4, s=20, color='darkorange')
axes[1].axhline(0, color='blue', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title(f'PCA space (PC1 explains {pca_2d.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.suptitle('PCA Finds the Axes of Maximum Variance', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca_2d.explained_variance_ratio_.round(3))

---
## Demo 2 — Explained Variance & Choosing Components

The **scree plot** (explained variance vs component number) shows how much information each component carries. Look for an 'elbow' — after which adding more components gives diminishing returns.

In [ ]:
wine = load_wine()
X_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = wine.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_wine)

pca_full = PCA().fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='darkorange')
axes[1].axhline(0.90, color='red', linestyle='--', label='90% variance')
axes[1].axhline(0.95, color='green', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 90% variance: {n_90}  (from {X_wine.shape[1]} original features)")
print(f"Components for 95% variance: {n_95}")

---
## Demo 3 — PCA Loadings: What Does Each Component Represent?

**Loadings** (component vectors) tell us which original features each PC captures.  
Large positive/negative loadings → that feature strongly influences the component.

In [ ]:
pca_3 = PCA(n_components=3)
pca_3.fit(X_scaled)

loadings = pd.DataFrame(
    pca_3.components_.T,
    index=wine.feature_names,
    columns=['PC1', 'PC2', 'PC3']
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (ax, pc) in enumerate(zip(axes, ['PC1', 'PC2', 'PC3'])):
    loadings[pc].sort_values().plot(kind='barh', ax=ax,
        color=loadings[pc].sort_values().apply(lambda x: 'steelblue' if x > 0 else 'salmon'),
        edgecolor='white')
    ev = pca_3.explained_variance_ratio_[i] * 100
    ax.set_title(f'{pc}  ({ev:.1f}% variance)', fontsize=11)
    ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('PCA Component Loadings — Wine Dataset', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Demo 4 — PCA for Dimensionality Reduction (Classification)

Compare Logistic Regression on original 13 features vs compressed PCA representation.

In [ ]:
results = {}
for n in [2, 3, 5, 8, 13]:
    pca_n = PCA(n_components=n)
    X_pca_n = pca_n.fit_transform(X_scaled)
    cv = cross_val_score(LogisticRegression(max_iter=500), X_pca_n, y_wine, cv=5)
    results[f'{n} components'] = cv.mean()

# Baseline on all features
cv_base = cross_val_score(LogisticRegression(max_iter=500), X_scaled, y_wine, cv=5)
results['All 13 features (baseline)'] = cv_base.mean()

print("Logistic Regression CV Accuracy:")
for k, v in results.items():
    print(f"  {k:35s}: {v:.4f}")

---
## When to Use PCA

| Situation | Use PCA? |
|---|---|
| Many highly correlated features | ✅ Yes — removes redundancy |
| Visualising high-dimensional data in 2D/3D | ✅ Yes |
| Reducing storage / computation cost | ✅ Yes |
| Features are independent / uncorrelated | ❌ Little benefit |
| Need interpretable features | ❌ Components are abstract combinations |
| Non-linear structure in data | ❌ Consider kernel PCA or UMAP instead |

---

## Key Takeaways

- PCA rotates the data to align axes with maximum variance — no information is discarded in the rotation itself
- **Standardise first** — always
- Use the **scree plot** to decide how many components to keep
- **Loadings** reveal what each component measures
- Fit PCA on **training data only**, transform both train and test

---

## Further Reading

- [Kaggle Tutorial — PCA](https://www.kaggle.com/code/ryanholbrook/principal-component-analysis)
- [StatQuest — PCA clearly explained](https://www.youtube.com/watch?v=FgakZw6K1QQ) (YouTube — highly recommended)
- [sklearn docs — PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Making sense of PCA, eigenvectors & eigenvalues](https://stats.stackexchange.com/questions/2691/making-sense-of-principal-component-analysis-eigenvectors-eigenvalues) — Cross Validated
- [UMAP: Uniform Manifold Approximation](https://umap-learn.readthedocs.io/) — non-linear alternative